In [0]:
%sql
USE CATALOG V_Commerce;

CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

In [0]:
def analisar_null_coluna(tabela):
    tabela.select([

        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in tabela.columns
        
    ]).display()

In [0]:
catalog = "V_Commerce"
silver_schema_name = "silver"

In [0]:
tb_avaliacoes_bronze = spark.table("v_commerce.bronze.tb_avaliacoes")
tb_catalogo_produtos_bronze = spark.table("v_commerce.bronze.tb_catalogo_produtos")
tb_clicktream_bronze = spark.table("v_commerce.bronze.tb_clickstream")
tb_cliente_bronze = spark.table("v_commerce.bronze.tb_clientes")
tb_pedidos_bronze = spark.table("v_commerce.bronze.tb_pedidos")
tb_suporte_tickets_bronze = spark.table("v_commerce.bronze.tb_suporte_tickets")

In [0]:
#tb_pedidos_bronze.display()
#analisar_null_coluna(tb_pedidos_bronze)

In [0]:
tb_pedidos_bronze.filter(
    F.expr("try_cast(quantidade AS INT)") < 0
).count()

8828

In [0]:
tb_pedidos_bronze.filter(
    F.expr("try_cast(valor_pedido AS FLOAT)") < 0
).count()

6246

In [0]:
tb_pedidos = (
    spark.table("v_commerce.bronze.tb_pedidos")

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_pedido").orderBy(F.col("timestamp_ingestion").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),

        F.round(
            F.when(
                F.regexp_replace(
                    F.regexp_replace(F.col("valor_pedido"), r"R\$|\s+", ""),
                    r",", "."
                ).cast(DoubleType()) < 0,
                None
            ).otherwise(
                F.regexp_replace(
                    F.regexp_replace(F.col("valor_pedido"), r"R\$|\s+", ""),
                    r",", "."
                ).cast(DoubleType())
            ), 2
        ).alias("valor_pedido"),

        F.coalesce(
            F.expr("try_to_date(data_pedido, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_pedido, 'dd-MM-yyyy')"),
            F.expr("try_to_date(data_pedido, 'MM-dd-yyyy')"),
            F.expr("try_to_date(data_pedido, 'dd/MM/yyyy')"),
            F.expr("try_to_date(data_pedido, 'MM/yyyy/dd')"),
            F.expr("try_to_date(data_pedido, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_pedido, 'yyyy/dd/MM')"),
            F.expr("try_to_date(data_pedido, 'dd-yyyy-MM')"),
            F.expr("try_to_date(data_pedido, 'MM-yyyy-dd')"),
        ).alias("data_pedido"),

        F.when(F.upper(F.col("metodo_pagamento")).rlike(r"^P[1I]X$"),                    "PIX")
         .when(F.upper(F.col("metodo_pagamento")).rlike(r"^B[0O]L(ET[0O])?$"),           "Boleto")
         .when(F.upper(F.col("metodo_pagamento")).rlike(r"^C[@A]RT([AÃ][OÃ])?$|^CRT$"), "Cartao")
         .otherwise(None)
         .alias("metodo_pagamento"),

        F.when(F.upper(F.col("status")).rlike(r"^APROV(ADO{1,2})?$|^APR$"),          "Aprovado")
         .when(F.upper(F.col("status")).rlike(r"^RECUS(ADO{1,2})?$|^REC$"),          "Recusado")
         .when(F.upper(F.col("status")).rlike(r"^PROC(ESS(ANDO)?)?$"),               "Processando")
         .when(F.upper(F.col("status")).rlike(r"^REEMB(OLS(O|AD[OA]?)?)?$"),         "Reembolsado")
         .otherwise(None)
         .alias("status"),

        F.when(F.col("quantidade") == "um",     1)
         .when(F.col("quantidade") == "dois",   2)
         .when(F.col("quantidade") == "tres",   3)
         .when(F.col("quantidade") == "quatro", 4)
         .when(F.col("quantidade") == "cinco",  5)
         .when(F.expr("try_cast(try_cast(quantidade AS FLOAT) AS INT)") < 0, None)
         .otherwise(F.expr("try_cast(try_cast(quantidade AS FLOAT) AS INT)"))
         .alias("quantidade"),

        F.to_timestamp(F.col("timestamp_ingestion")).alias("timestamp_ingestion"),
    )
)

tb_pedidos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("v_commerce.silver.tb_pedidos")
print("✅ Tabela silver.tb_pedidos criada com sucesso!\n")

✅ Tabela silver.tb_pedidos criada com sucesso!



In [0]:
#tb_pedidos.display()
#analisar_null_coluna(tb_pedidos)

In [0]:
#tb_avaliacoes_bronze.display()
#analisar_null_coluna(tb_avaliacoes_bronze)

In [0]:
#tb_avaliacoes_bronze.select("data_avaliacao").distinct().show()


In [0]:
# prata.tb_avaliacoes
# Origem: bronze.tb_avaliacoes

tb_avaliacoes = (
    spark.table("v_commerce.bronze.tb_avaliacoes")

    .withColumn(
        "_rank",
        F.row_number().over(
            Window.partitionBy("id_avaliacao").orderBy(F.col("timestamp_ingestion").desc())
        )
    )
    .filter(F.col("_rank") == 1)
    .drop("_rank")

    .select(
        F.col("id_avaliacao"),
        F.col("id_pedido"),
        F.col("id_cliente"),
        F.col("id_produto"),

        F.when(F.col("nota_produto") == "péssimo", 1)
         .when(F.col("nota_produto") == "ruim",    2)
         .when(F.col("nota_produto") == "bom",     4)
         .when(F.col("nota_produto") == "ótimo",   5)
         .when(F.expr("try_cast(nota_produto AS INT)").between(1, 5),
               F.expr("try_cast(nota_produto AS INT)"))
         .otherwise(-1)
         .alias("nota_produto"),

        F.col("comentario"),

        F.when(F.col("nota_nps") == "péssimo", 1)
         .when(F.col("nota_nps") == "ruim",    3)
         .when(F.col("nota_nps") == "bom",     8)
         .when(F.col("nota_nps") == "ótimo",   10)
         .when(F.expr("try_cast(nota_nps AS INT)").between(0, 10),
               F.expr("try_cast(nota_nps AS INT)"))
         .otherwise(-1)
         .alias("nota_nps"),

        F.when(F.lower(F.col("recomenda")).isin("s", "sim", "yes", "1"), "Sim")
         .when(F.lower(F.col("recomenda")).isin("n", "nao", "não", "no", "0"), "Não")
         .otherwise(None)
         .alias("recomenda"),

        F.coalesce(
            # com timestamp
            F.expr("try_to_date(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/MM/dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd-MM-yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM-dd-yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd/MM/yyyy HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM/yyyy/dd HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'dd-yyyy-MM HH:mm:ss')"),
            F.expr("try_to_date(data_avaliacao, 'MM-yyyy-dd HH:mm:ss')"),
            # sem timestamp
            F.expr("try_to_date(data_avaliacao, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/dd/MM')"),
            F.expr("try_to_date(data_avaliacao, 'yyyy/MM/dd')"),
            F.expr("try_to_date(data_avaliacao, 'dd-MM-yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'MM-dd-yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'dd/MM/yyyy')"),
            F.expr("try_to_date(data_avaliacao, 'MM/yyyy/dd')"),
            F.expr("try_to_date(data_avaliacao, 'dd-yyyy-MM')"),
            F.expr("try_to_date(data_avaliacao, 'MM-yyyy-dd')"),
        ).alias("data_avaliacao"),

        F.to_timestamp(F.col("timestamp_ingestion")).alias("timestamp_ingestion"),
    )
)

tb_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("v_commerce.silver.tb_avaliacoes")
print("✅ Tabela silver.tb_avaliacoes criada com sucesso!\n")

✅ Tabela silver.tb_avaliacoes criada com sucesso!



In [0]:
#tb_avaliacoes.display()
#analisar_null_coluna(tb_avaliacoes)